<a href="https://colab.research.google.com/github/YaninaColangelo/market_labor_analysis/blob/main/notebooks/00_1_exploracion_fuente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import requests
from bs4 import BeautifulSoup

from pathlib import Path
import sys

root = Path.cwd()
if not (root / "src").exists():
    root = root.parent

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [3]:
from src.connectors.tecnoempleo import construir_oferta_raw
from src.etl import transformar_oferta
from src.data_collection import RAW_COLUMNS, load_raw_dataset, append_offer, save_raw_dataset

In [5]:
!git clone https://github.com/YaninaColangelo/market_labor_analysis.git

Cloning into 'market_labor_analysis'...


In [6]:
%cd market_labor_analysis

c:\Users\yanin\Documents\market_labor_analysis\notebooks\market_labor_analysis


# **Extractor**

# 1.1 — Conectividad

In [7]:
url = "https://www.tecnoempleo.com/analista-datos-powerbi-sopra-steria/power-bi-etl/rf-7624176cc2dbe32ca545"

headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)

print(response.status_code)

403


# 1.2 — Crear el objeto BeautifulSoup

In [8]:
soup = BeautifulSoup(response.text, "html.parser")

# 1.3 — Primera extracción

In [9]:
meta_title = soup.find("meta", property="og:title")

print(meta_title)

None


In [10]:
meta_description = soup.find("meta",attrs={"name": "description"})

print(meta_description)

None


# 1.4 — Validación

In [ ]:
descripcion_meta = meta_description.get("content") if meta_description else None

print(descripcion_meta)

Oferta de Empleo analista de datos powerbi en Madrid, Sopra Steria - Tecnoempleo.com con conocimientos power bi, etl, sql server


# 1.5 — Oferta extraida desde la fuente

In [ ]:
# Esta estructura todavia conserva campos propios de la extraccion web.

oferta_extraida = construir_oferta_raw(response.text, url)

oferta_extraida

# **Transformador**

# Primera transformación

In [ ]:
# Transformacion central del repo:
oferta_transformada = transformar_oferta(oferta_extraida)

oferta_transformada

In [ ]:
# Adaptacion final al esquema raw del proyecto.

oferta_raw = {
    column: oferta_transformada.get(column)
    for column in RAW_COLUMNS
}

oferta_raw

Analista de datos PowerBI
Madrid


# Segunda transformación
Actualizar el diccionario.

In [ ]:
# El id_oferta queda vacio para que data_collection lo asigne
oferta_raw["id_oferta"] = None

oferta_raw

# **Loader**

In [ ]:
# Cargar el dataset raw existente.

df_raw = load_raw_dataset()

df_raw.tail()

In [ ]:
# Evitar duplicar la misma oferta si el notebook se ejecuta mas de una vez.

df_raw = df_raw[df_raw["url"] != oferta_raw["url"]].copy()

df_raw = append_offer(df_raw, oferta_raw)

save_raw_dataset(df_raw)

df_raw.tail()

In [ ]:
# Comprobar que la oferta extraida quedo incorporada al dataset raw
# con el mismo formato que las ofertas de prueba.

df_raw[df_raw["url"] == url]

✔ Se eligió la fuente (Tecnoempleo).

✔ Se verificó el acceso mediante requests.

✔ Se parseó el HTML con BeautifulSoup.

✔ Se extrajeron los metadatos.

✔ Se diseñó el diccionario estándar del proyecto.

✔ Se implementó la primera transformación del ETL (titulo_puesto y ciudad).

### Ya validamos el primer bloque del pipeline.